## Advanced Tutorial: Performing Byte-Range Requests On GRIB Files Using The Raw Client `byte_range_request()` From WxData

### What is a byte-range request and why is this useful?

It is useful to perform byte-range requests when there is no GRIB filter for the data server.

***TLDR: Byte-Range Requests minimize both download time and traffic over the network. This is very useful when there is no GRIB filter present.***

Byte-range requests allow for the user to subset the data by variable, level (pressure, height, depth) and level type (i.e. 'pressure' or 'height above ground' etc.)

Variables in GRIB filter are organized by the range in bytes within the large GRIB file. A bytes-range request is when we perform a partial download
by scanning the index (.idx) file for the model data and then using the byte-ranges found in that index file for the variables we are requesting as headers in our HTTPS request. This allows us to only download the variables we need rather than download the entire large file.

**WxData >= 1.7 makes this easy**

In this example, we will download the first 72 hours of the latest GFS data using byte-range requests instead of the GRIB filter. 

***GFS Variables***

    'best lifted index'
    'absolute vorticity'
    'convective precipitation'
    'albedo'
    'total precipitation'
    'convective available potential energy'
    'categorical freezing rain'
    'categorical ice pellets'
    'convective inhibition'
    'cloud mixing ratio'
    'plant canopy surface water'
    'percent frozen precipitaion'
    'convective precipitation rate'
    'categorical rain'
    'categorical snow'
    'cloud water'
    'cloud work function'
    'downward longwave radiation flux'
    'dew point'
    'downward shortwave radiation flux'
    'vertical velocity (height)'
    'field capacity'
    'surface friction velocity'
    'ground heat flux'
    'graupel'
    'wind gust'
    'high cloud cover'
    'geopotential height'
    'haines index'
    'storm relative helicity'
    'planetary boundary layer height'
    'icao standard atmosphere reference height'
    'ice cover'
    'ice growth rate'
    'ice thickness'
    'ice temperature'
    'ice water mixing ratio'
    'land cover'
    'low cloud cover'
    'surface lifted index'
    'latent heat net flux'
    'middle cloud cover'
    'mslp (eta model reduction)'
    'ozone mixing ratio'
    'potential evaporation rate'
    'pressure level from which parcel was lifted'
    'potential temperature'
    'precipitation rate'
    'pressure'
    'mean sea level pressure'
    'precipitable water'
    'composite reflectivity'
    'reflectivity'
    'relative humidity'
    'rain mixing ratio'
    'surface roughness'
    'sensible heat net flux'
    'snow mixing ratio'
    'snow depth'
    'liquid volumetric soil moisture (non-frozen)'
    'volumetric soil moisture content'
    'soil type'
    'specific humidity'
    'sunshine duration'
    'total cloud cover'
    'maximum temperature'
    'minimum temperature'
    'temperature'
    'total ozone'
    'soil temperature'
    'momentum flux (u-component)'
    'u-component of wind'
    'zonal flux of gravity wave stress'
    'upward longwave radiation flux'
    'u-component of storm motion'
    'upward shortwave radiation flux'
    'vegetation'
    'momentum flux (v-component)'
    'v-component of wind'
    'meridional flux of gravity wave stress'
    'visibility'
    'ventilation rate'
    'v-component of storm motion'
    'vertical velocity (pressure)'
    'vertical speed shear'
    'water runoff'
    'water equivalent of accumulated snow depth'
    'wilting point'  

***GFS Level Types***

    'hybrid'
    'entire atmosphere'
    'surface':'surface',
    'boundary layer'
    'pressure'
    'mean sea level'
    'height above ground'
    'height below ground'
    'height above sea level'
    'entire atmosphere single layer'
    'low cloud layer'
    'middle cloud layer'
    'high cloud layer'
    'cloud ceiling'
    'tropopause'
    'max wind'
    'isothermal'
    'highest tropospheric freezing level'
    'sigma layer'
    'sigma level'
    'potential vorticity surface'

If you are getting SSL Certificate Errors when trying to download data, you may be using a proxy server connection

We will not be doing any proxy connection examples.

How to set up a proxy:

proxies (dict or None) - Default=None. If the user is using proxy server(s), the user must change the following:

   proxies=None ---> proxies={
                           'http':'http://your-proxy-address:port',
                           'https':'http://your-proxy-address:port'
                           }
For more information on configuring proxies: https://requests.readthedocs.io/en/latest/user/advanced/#proxies

#### Imports

In [1]:
import wxdata.client as client
import xarray as xr
import os

from datetime import datetime, timedelta

#### Getting Our Model Runtime

In [2]:
utc = datetime.utcnow()
local = datetime.now()

if utc.hour >= 3 and utc.hour < 9:
    hour = '00'
    date = utc
elif utc.hour >= 9 and utc.hour < 15:
    hour = '06'
    date = utc
elif utc.hour >= 15 and utc.hour < 21:
    hour = '12'
    date = utc
else:
    if utc.day == local.day:
        hour = '18'
        date = utc
    else:
        hour = '18'
        date = local

#### Making Our Bytes-Range Request

##### Pressure Levels

We are now making our bytes-range request for the following information:

Variables: temperature, relative humidity, geopotential height, u-wind and v-wind. 

Levels: 925mb, 850mb, 700mb, 500mb, 300mb, 200mb, 10mb

In [3]:
variables = ['temperature', 'relative humidity', 'geopotential height', 'u-component of wind', 'v-component of wind']
levels = [925, 850, 700, 500, 300, 250, 200, 10]
level_type = 'pressure'
path = f"GFS/Pressure"

for i in range(0, 75, 3):

    if i < 10:
        client.byte_range_request(f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f00{i}",
                                  f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f00{i}.idx",
                                  variables,
                                  levels,
                                  level_type,
                                  path,
                                  f"gfs.t{hour}z.pgrb2.0p25.f00{i}.grib2")
    else:
        client.byte_range_request(f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f0{i}",
                                  f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f0{i}.idx",
                                  variables,
                                  levels,
                                  level_type,
                                  path,
                                  f"gfs.t{hour}z.pgrb2.0p25.f0{i}.grib2")

Writing GRIB messages     |                              | 0/40

HGT @ 10 mb                    |                              |   0% [0.00/668k]

HGT @ 200 mb                   |                              |   0% [0.00/723k]

HGT @ 250 mb                   |                              |   0% [0.00/717k]

HGT @ 300 mb                   |                              |   0% [0.00/710k]

...

VGRD @ 500 mb                  |                              |   0% [0.00/548k]

VGRD @ 700 mb                  |                              |   0% [0.00/902k]

VGRD @ 850 mb                  |                              |   0% [0.00/941k]

VGRD @ 925 mb                  |                              |   0% [0.00/949k]

gfs.t00z.pgrb2.0p25.f072.grib2 saved to GFS/Pressure


##### Height Above Ground

We are now making our bytes-range request for the following information:

Variables: temperature and relative humidity. 

Levels: 2-meter

In [4]:
variables = ['temperature', 'relative humidity']
levels = [2]
level_type = 'height above ground'
path = f"GFS/Height Above Ground"

for i in range(0, 75, 3):

    if i < 10:
        client.byte_range_request(f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f00{i}",
                                  f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f00{i}.idx",
                                  variables,
                                  levels,
                                  level_type,
                                  path,
                                  f"gfs.t{hour}z.pgrb2.0p25.f00{i}.grib2")
    else:
        client.byte_range_request(f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f0{i}",
                                  f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f0{i}.idx",
                                  variables,
                                  levels,
                                  level_type,
                                  path,
                                  f"gfs.t{hour}z.pgrb2.0p25.f0{i}.grib2")

Writing GRIB messages     |                              | 0/2

RH @ 2 m above ground          |                              |   0% [0.00/770k]

TMP @ 2 m above ground         |                              |   0% [0.00/494k]

gfs.t00z.pgrb2.0p25.f000.grib2 saved to GFS/Height Above Ground


Writing GRIB messages     |                              | 0/2

...

gfs.t00z.pgrb2.0p25.f069.grib2 saved to GFS/Height Above Ground


Writing GRIB messages     |                              | 0/2

RH @ 2 m above ground          |                              |   0% [0.00/776k]

TMP @ 2 m above ground         |                              |   0% [0.00/496k]

gfs.t00z.pgrb2.0p25.f072.grib2 saved to GFS/Height Above Ground


##### Single Levels (Where levels=None)

We are now making our bytes-range request for the following information:

Variables: temperature and geopotential height.

In this case, we are looking at the max wind layer. 
Since this is a single layer, we must set levels=None because there is no array of numerical values to represent the level. 

In other words, this is the max wind layer in the entire atmosphere. There is no set value (i.e. 500 mb) that represents this level.

In [5]:
variables = ['temperature', 'geopotential height']
levels = None
level_type = 'max wind'
path = f"GFS/Max Wind"

for i in range(0, 75, 3):

    if i < 10:
        client.byte_range_request(f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f00{i}",
                                  f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f00{i}.idx",
                                  variables,
                                  levels,
                                  level_type,
                                  path,
                                  f"gfs.t{hour}z.pgrb2.0p25.f00{i}.grib2")
    else:
        client.byte_range_request(f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f0{i}",
                                  f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date.strftime('%Y%m%d')}/{hour}/atmos/gfs.t{hour}z.pgrb2.0p25.f0{i}.idx",
                                  variables,
                                  levels,
                                  level_type,
                                  path,
                                  f"gfs.t{hour}z.pgrb2.0p25.f0{i}.grib2")

Writing GRIB messages     |                              | 0/2

HGT @ max wind                 |                              |   0% [0.00/1.62M]

TMP @ max wind                 |                              |   0% [0.00/1.21M]

gfs.t00z.pgrb2.0p25.f000.grib2 saved to GFS/Max Wind


Writing GRIB messages     |                              | 0/2

...

gfs.t00z.pgrb2.0p25.f069.grib2 saved to GFS/Max Wind


Writing GRIB messages     |                              | 0/2

HGT @ max wind                 |                              |   0% [0.00/1.60M]

TMP @ max wind                 |                              |   0% [0.00/1.21M]

gfs.t00z.pgrb2.0p25.f072.grib2 saved to GFS/Max Wind


#### Getting Our File Paths

In [6]:
pressure_data_paths = []
for file in os.listdir(f"GFS/Pressure"):
    path = f"GFS/Pressure/{file}"
    pressure_data_paths.append(path)

In [7]:
pressure_data_paths = sorted(pressure_data_paths)

In [8]:
pressure_data_paths

['GFS/Pressure/gfs.t00z.pgrb2.0p25.f000.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f003.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f006.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f009.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f012.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f015.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f018.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f021.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f024.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f027.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f030.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f033.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f036.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f039.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f042.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f045.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f048.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f051.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f054.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f057.grib2',
 'GFS/Pressure/gfs.t00z.pgrb2.0p25.f060.

In [9]:
height_data_paths = []
for file in os.listdir(f"GFS/Height Above Ground"):
    path = f"GFS/Height Above Ground/{file}"
    height_data_paths.append(path)

In [10]:
height_data_paths = sorted(height_data_paths)

In [11]:
height_data_paths

['GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f000.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f003.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f006.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f009.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f012.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f015.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f018.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f021.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f024.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f027.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f030.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f033.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f036.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f039.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f042.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f045.grib2',
 'GFS/Height Above Ground/gfs.t00z.pgrb2.0p25.f048.grib2

In [12]:
max_wind_data_paths = []
for file in os.listdir(f"GFS/Max Wind"):
    path = f"GFS/Max Wind/{file}"
    max_wind_data_paths.append(path)

In [13]:
max_wind_data_paths = sorted(max_wind_data_paths)

In [14]:
max_wind_data_paths

['GFS/Max Wind/gfs.t00z.pgrb2.0p25.f000.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f003.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f006.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f009.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f012.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f015.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f018.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f021.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f024.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f027.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f030.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f033.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f036.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f039.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f042.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f045.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f048.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f051.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f054.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f057.grib2',
 'GFS/Max Wind/gfs.t00z.pgrb2.0p25.f060.

#### Opening Our Data in Xarray

We will now use `xr.open_mfdataset()` to open all the files in each folder and combine it into one xarray.array

In [15]:
# Pressure Level Data
pressure_ds = xr.open_mfdataset(pressure_data_paths,
                              concat_dim='step', 
                              combine='nested', 
                              coords='minimal', 
                              engine='cfgrib', 
                              compat='override', 
                              decode_timedelta=False,
                              backend_kwargs={"indexpath": ""})

In [16]:
pressure_ds

<xarray.Dataset> Size: 4GB
Dimensions:        (step: 25, isobaricInhPa: 8, latitude: 721, longitude: 1440)
Coordinates:
    time           datetime64[ns] 8B ...
  * step           (step) float64 200B 0.0 3.0 6.0 9.0 ... 63.0 66.0 69.0 72.0
  * isobaricInhPa  (isobaricInhPa) float64 64B 925.0 850.0 700.0 ... 200.0 10.0
  * latitude       (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude      (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
    valid_time     datetime64[ns] 8B ...
Data variables:
    gh             (step, isobaricInhPa, latitude, longitude) float32 831MB dask.array<chunksize=(1, 8, 721, 1440), meta=np.ndarray>
    r              (step, isobaricInhPa, latitude, longitude) float32 831MB dask.array<chunksize=(1, 8, 721, 1440), meta=np.ndarray>
    t              (step, isobaricInhPa, latitude, longitude) float32 831MB dask.array<chunksize=(1, 8, 721, 1440), meta=np.ndarray>
    u              (step, isobaricInhPa, latitude, longitude) float32 831MB dask.array<chunksize=(1, 8, 721, 1440), meta=np.ndarray>
    v              (step, isobaricInhPa, latitude, longitude) float32 831MB dask.array<chunksize=(1, 8, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-05-07T23:04 GRIB to CDM+CF via cfgrib-0.9.1...

In [17]:
# Height Level Data
height_ds = xr.open_mfdataset(height_data_paths,
                              concat_dim='step', 
                              combine='nested', 
                              coords='minimal', 
                              engine='cfgrib', 
                              compat='override', 
                              decode_timedelta=False,
                              backend_kwargs={"indexpath": ""})

In [18]:
height_ds

<xarray.Dataset> Size: 208MB
Dimensions:            (step: 25, latitude: 721, longitude: 1440)
Coordinates:
    time               datetime64[ns] 8B ...
  * step               (step) float64 200B 0.0 3.0 6.0 9.0 ... 66.0 69.0 72.0
    heightAboveGround  float64 8B ...
  * latitude           (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude          (longitude) float64 12kB 0.0 0.25 0.5 ... 359.5 359.8
    valid_time         datetime64[ns] 8B ...
Data variables:
    r2                 (step, latitude, longitude) float32 104MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t2m                (step, latitude, longitude) float32 104MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-05-07T23:04 GRIB to CDM+CF via cfgrib-0.9.1...

In [19]:
# Height Level Data
max_wind_ds = xr.open_mfdataset(max_wind_data_paths,
                              concat_dim='step', 
                              combine='nested', 
                              coords='minimal', 
                              engine='cfgrib', 
                              compat='override', 
                              decode_timedelta=False,
                              backend_kwargs={"indexpath": ""})

In [20]:
max_wind_ds

<xarray.Dataset> Size: 208MB
Dimensions:     (step: 25, latitude: 721, longitude: 1440)
Coordinates:
    time        datetime64[ns] 8B ...
  * step        (step) float64 200B 0.0 3.0 6.0 9.0 12.0 ... 63.0 66.0 69.0 72.0
    maxWind     float64 8B ...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    valid_time  datetime64[ns] 8B ...
Data variables:
    gh          (step, latitude, longitude) float32 104MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    t           (step, latitude, longitude) float32 104MB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-05-07T23:04 GRIB to CDM+CF via cfgrib-0.9.1...